In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models
import numpy as np
from tensorflow.keras.callbacks import TensorBoard
import datetime

# 加载并预处理CIFAR-10数据集（你的原有代码）
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.cifar10.load_data()
x_train = x_train.astype('float32') / 255.0
x_test = x_test.astype('float32') / 255.0
mean = np.mean(x_train, axis=(0, 1, 2))
x_train -= mean
x_test -= mean
y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# 定义basic_block（你的原有代码）
def basic_block(x, out_channels, stride=1, use_bn=True):
    residual = x
    x = layers.Conv2D(out_channels, (3, 3), strides=stride, padding='same', use_bias=False)(x)
    if use_bn:
        x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = layers.Conv2D(out_channels, (3, 3), strides=1, padding='same', use_bias=False)(x)
    if use_bn:
        x = layers.BatchNormalization()(x)
    if stride != 1 or residual.shape[-1] != out_channels:
        residual = layers.Conv2D(out_channels, (1, 1), strides=stride, use_bias=False)(residual)
        if use_bn:
            residual = layers.BatchNormalization()(residual)
    x = layers.Add()([x, residual])
    x = layers.ReLU()(x)
    return x

# 构建模型（你的原有代码）
def build_resnet():
    inputs = layers.Input(shape=(32, 32, 3))
    x = layers.Conv2D(16, (3, 3), padding='same', use_bias=False)(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    x = basic_block(x, 16, stride=1, use_bn=True)
    x = basic_block(x, 16, stride=1, use_bn=True)
    x = basic_block(x, 32, stride=2, use_bn=True)
    x = basic_block(x, 32, stride=1, use_bn=True)
    x = basic_block(x, 64, stride=2, use_bn=True)
    x = basic_block(x, 64, stride=1, use_bn=True)
    x = layers.GlobalAveragePooling2D()(x)
    outputs = layers.Dense(10, activation='softmax')(x)
    model = models.Model(inputs=inputs, outputs=outputs)
    return model

# 编译模型
model = build_resnet()
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# 创建TensorBoard回调
log_dir = "logs/cifar10_resnet/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(
    log_dir=log_dir,
    histogram_freq=1,
    write_graph=True
)

# 训练模型并加入回调
model.fit(
    x_train, y_train,
    epochs=20,
    batch_size=64,
    validation_data=(x_test, y_test),
    callbacks=[tensorboard_callback]
)

# 保存模型时，建议改用Keras原生格式（避免HDF5警告）
model.save("my_resnet.keras")

2025-11-13 18:59:55.288777: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-11-13 18:59:55.288825: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-11-13 18:59:55.290027: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


Epoch 1/20


I0000 00:00:1763031595.230588    7028 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


782/782 [==============================] - 8s 4ms/step - loss: 1.3065 - accuracy: 0.5264 - val_loss: 1.1517 - val_accuracy: 0.5925
Epoch 2/20
782/782 [==============================] - 8s 11ms/step - loss: 0.9203 - accuracy: 0.6737 - val_loss: 0.9469 - val_accuracy: 0.6603
Epoch 3/20
782/782 [==============================] - 9s 11ms/step - loss: 0.7550 - accuracy: 0.7352 - val_loss: 1.0756 - val_accuracy: 0.6389
Epoch 4/20
782/782 [==============================] - 10s 12ms/step - loss: 0.6494 - accuracy: 0.7725 - val_loss: 1.1078 - val_accuracy: 0.6528
Epoch 5/20
782/782 [==============================] - 11s 14ms/step - loss: 0.5753 - accuracy: 0.7987 - val_loss: 0.7232 - val_accuracy: 0.7502
Epoch 6/20
782/782 [==============================] - 12s 15ms/step - loss: 0.5044 - accuracy: 0.8237 - val_loss: 0.8336 - val_accuracy: 0.7156
Epoch 7/20
782/782 [==============================] - 13s 17ms/step - loss: 0.4529 - accuracy: 0.8429 - val_loss: 0.8665 - val_accuracy: 0.7212
Epoch 8